In [4]:
import os
import json
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

In [5]:
# Update these paths to where your validation dataset is located
# (e.g., if you are running this locally vs on Kaggle)
VAL_IMG = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/validation/image"
VAL_ANNO = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/validation/annos"

# Point this to your trained `cls.pth` model
MODEL_WEIGHTS = "/kaggle/input/models/parrvvv/resnet/pytorch/default/1/cls.pth"

NUM_CLASSES = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
# Labels logic
category_map = {1: 0, 2: 1, 7: 2, 8: 3, 9: 4}
class_names = ["short sleeve top", "long sleeve top", "shorts", "trousers", "skirt"]

def get_labels(json_path):
    with open(json_path) as f:
        data = json.load(f)
        
    label = np.zeros(NUM_CLASSES)
    for k in data:
        if "item" in k:
            cat = data[k]["category_id"]
            if cat in category_map:
                label[category_map[cat]] = 1
    return label

In [7]:
# Validation Transform matching your predictor logic
val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ValidationDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, anno_dir, transform):
        self.img_dir = img_dir
        self.anno_dir = anno_dir
        self.images = sorted(os.listdir(img_dir))[:5000] # Use only the first 5000 images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        anno_path = os.path.join(self.anno_dir, img_name.replace(".jpg", ".json"))
        
        image = Image.open(img_path).convert("RGB")
        label = get_labels(anno_path)
        
        image_tensor = self.transform(image)
        return image_tensor, torch.tensor(label).float()

val_dataset = ValidationDataset(VAL_IMG, VAL_ANNO, transform=val_transform)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
print(f"Loaded validation set: {len(val_dataset)} images")

Loaded validation set: 5000 images


In [8]:
# Load the Model
model = models.resnet50(weights=None)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)

print(f"Loading weights from {MODEL_WEIGHTS}...")
model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=device))
model.to(device)
model.eval()
print("Model loaded successfully!")

Loading weights from /kaggle/input/models/parrvvv/resnet/pytorch/default/1/cls.pth...
Model loaded successfully!


In [9]:
# Evaluation Loop
all_preds = []
all_targets = []

with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc="Running Inference"):
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        outputs = model(imgs)
        probs = torch.sigmoid(outputs)
        
        # Threshold at 0.5
        preds = (probs > 0.5).float()
        
        all_preds.append(preds.cpu().numpy())
        all_targets.append(labels.cpu().numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

Running Inference: 100%|██████████| 157/157 [00:45<00:00,  3.46it/s]


In [10]:
# Calculate and display metrics
macro_prec = precision_score(all_targets, all_preds, average="macro", zero_division=0)
micro_prec = precision_score(all_targets, all_preds, average="micro", zero_division=0)

macro_rec = recall_score(all_targets, all_preds, average="macro", zero_division=0)
micro_rec = recall_score(all_targets, all_preds, average="micro", zero_division=0)

macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
micro_f1 = f1_score(all_targets, all_preds, average="micro", zero_division=0)

print("========== OVERALL METRICS ==========")
print(f"Macro Precision:  {macro_prec:.4f}")
print(f"Micro Precision:  {micro_prec:.4f}")
print("-" * 37)
print(f"Macro Recall:     {macro_rec:.4f}")
print(f"Micro Recall:     {micro_rec:.4f}")
print("-" * 37)
print(f"Macro F1-Score:   {macro_f1:.4f}")
print(f"Micro F1-Score:   {micro_f1:.4f}")
print("=====================================\n")

print("========== PER-CLASS REPORT =========")
print(classification_report(all_targets, all_preds, target_names=class_names, zero_division=0))

========== OVERALL METRICS ==========
Macro Precision:  0.5065
Micro Precision:  0.2668
-------------------------------------
Macro Recall:     0.2209
Micro Recall:     0.2067
-------------------------------------
Macro F1-Score:   0.1162
Micro F1-Score:   0.2329

========== PER-CLASS REPORT =========
                  precision    recall  f1-score   support

short sleeve top       1.00      0.00      0.01      2300
 long sleeve top       0.27      0.11      0.15      1388
          shorts       0.00      0.00      0.00       522
        trousers       0.27      0.99      0.42      1319
           skirt       1.00      0.00      0.00      1574

       micro avg       0.27      0.21      0.23      7103
       macro avg       0.51      0.22      0.12      7103
    weighted avg       0.65      0.21      0.11      7103
     samples avg       0.27      0.20      0.22      7103



In [11]:
# Update this to your MobileNet .pth weights if you have it!
MODEL_WEIGHTS_MOBILENET = "/kaggle/input/models/parrvvv/mobilenet/pytorch/default/1/mobilenet_transfer_best.pth"

# Let's initialize a MobileNetV3 Large
# (Change `mobilenet_v3_large` to `mobilenet_v3_small` if you trained the small version!)
mobilenet_model = models.mobilenet_v3_large(weights=None)
in_features_mn = mobilenet_model.classifier[3].in_features
mobilenet_model.classifier[3] = nn.Linear(in_features_mn, NUM_CLASSES)

if os.path.exists(MODEL_WEIGHTS_MOBILENET):
    print(f"Loading MobileNet weights from {MODEL_WEIGHTS_MOBILENET}...")
    mobilenet_model.load_state_dict(torch.load(MODEL_WEIGHTS_MOBILENET, map_location=device))
else:
    print(f"File not found: {MODEL_WEIGHTS_MOBILENET}. Skipping loading weights.")

mobilenet_model.to(device)
mobilenet_model.eval()
print("MobileNet Model initialized!")

Loading MobileNet weights from /kaggle/input/models/parrvvv/mobilenet/pytorch/default/1/mobilenet_transfer_best.pth...
MobileNet Model initialized!


In [12]:
# MobileNet Evaluation Loop
mn_preds = []
mn_targets = []

if os.path.exists(MODEL_WEIGHTS_MOBILENET):
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Running MobileNet Inference"):
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = mobilenet_model(imgs)
            probs = torch.sigmoid(outputs)

            # Threshold at 0.5
            preds = (probs > 0.5).float()

            mn_preds.append(preds.cpu().numpy())
            mn_targets.append(labels.cpu().numpy())

    mn_preds = np.concatenate(mn_preds)
    mn_targets = np.concatenate(mn_targets)

    # Calculate and display metrics
    macro_prec_mn = precision_score(mn_targets, mn_preds, average="macro", zero_division=0)
    micro_prec_mn = precision_score(mn_targets, mn_preds, average="micro", zero_division=0)

    macro_rec_mn = recall_score(mn_targets, mn_preds, average="macro", zero_division=0)
    micro_rec_mn = recall_score(mn_targets, mn_preds, average="micro", zero_division=0)

    macro_f1_mn = f1_score(mn_targets, mn_preds, average="macro", zero_division=0)
    micro_f1_mn = f1_score(mn_targets, mn_preds, average="micro", zero_division=0)

    print("========== MOBILENET METRICS ==========")
    print(f"Macro Precision:  {macro_prec_mn:.4f}")
    print(f"Micro Precision:  {micro_prec_mn:.4f}")
    print("-" * 37)
    print(f"Macro Recall:     {macro_rec_mn:.4f}")
    print(f"Micro Recall:     {micro_rec_mn:.4f}")
    print("-" * 37)
    print(f"Macro F1-Score:   {macro_f1_mn:.4f}")
    print(f"Micro F1-Score:   {micro_f1_mn:.4f}")
    print("=======================================\n")

    print("====== MOBILENET PER-CLASS REPORT =====")
    print(classification_report(mn_targets, mn_preds, target_names=class_names, zero_division=0))
else:
    print("Please set the variable `MODEL_WEIGHTS_MOBILENET` to your .pth file to evaluate.")

Running MobileNet Inference: 100%|██████████| 157/157 [00:21<00:00,  7.45it/s]

========== MOBILENET METRICS ==========
Macro Precision:  0.7200
Micro Precision:  0.7405
-------------------------------------
Macro Recall:     0.7694
Micro Recall:     0.7953
-------------------------------------
Macro F1-Score:   0.7427
Micro F1-Score:   0.7669

====== MOBILENET PER-CLASS REPORT =====
                  precision    recall  f1-score   support

short sleeve top       0.75      0.87      0.80      2300
 long sleeve top       0.71      0.66      0.69      1388
          shorts       0.60      0.66      0.63       522
        trousers       0.77      0.86      0.81      1319
           skirt       0.78      0.80      0.79      1574

       micro avg       0.74      0.80      0.77      7103
       macro avg       0.72      0.77      0.74      7103
    weighted avg       0.74      0.80      0.77      7103
     samples avg       0.76      0.80      0.76      7103



In [13]:
MODEL_WEIGHTS_EFFICIENTNET = "/kaggle/input/models/parrvvv/efficient-net/pytorch/default/1/best_eff_net_TL.pth"

# Let's initialize an EfficientNet B0
# (Change `efficientnet_b0` to `efficientnet_b1`, etc. if you trained a different version!)
efficientnet_model = models.efficientnet_b0(weights=None)
in_features_en = efficientnet_model.classifier[1].in_features
efficientnet_model.classifier[1] = nn.Linear(in_features_en, NUM_CLASSES)

if os.path.exists(MODEL_WEIGHTS_EFFICIENTNET):
    print(f"Loading EfficientNet weights from {MODEL_WEIGHTS_EFFICIENTNET}...")
    efficientnet_model.load_state_dict(torch.load(MODEL_WEIGHTS_EFFICIENTNET, map_location=device))
else:
    print(f"File not found: {MODEL_WEIGHTS_EFFICIENTNET}. Skipping loading weights.")

efficientnet_model.to(device)
efficientnet_model.eval()
print("EfficientNet Model initialized!")

Loading EfficientNet weights from /kaggle/input/models/parrvvv/efficient-net/pytorch/default/1/best_eff_net_TL.pth...
EfficientNet Model initialized!


In [14]:
# EfficientNet Evaluation Loop
en_preds = []
en_targets = []

if os.path.exists(MODEL_WEIGHTS_EFFICIENTNET):
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Running EfficientNet Inference"):
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = efficientnet_model(imgs)
            probs = torch.sigmoid(outputs)

            # Threshold at 0.5
            preds = (probs > 0.5).float()

            en_preds.append(preds.cpu().numpy())
            en_targets.append(labels.cpu().numpy())

    en_preds = np.concatenate(en_preds)
    en_targets = np.concatenate(en_targets)

    # Calculate and display metrics
    macro_prec_en = precision_score(en_targets, en_preds, average="macro", zero_division=0)
    micro_prec_en = precision_score(en_targets, en_preds, average="micro", zero_division=0)

    macro_rec_en = recall_score(en_targets, en_preds, average="macro", zero_division=0)
    micro_rec_en = recall_score(en_targets, en_preds, average="micro", zero_division=0)

    macro_f1_en = f1_score(en_targets, en_preds, average="macro", zero_division=0)
    micro_f1_en = f1_score(en_targets, en_preds, average="micro", zero_division=0)

    print("========== EFFICIENTNET METRICS ==========")
    print(f"Macro Precision:  {macro_prec_en:.4f}")
    print(f"Micro Precision:  {micro_prec_en:.4f}")
    print("-" * 37)
    print(f"Macro Recall:     {macro_rec_en:.4f}")
    print(f"Micro Recall:     {micro_rec_en:.4f}")
    print("-" * 37)
    print(f"Macro F1-Score:   {macro_f1_en:.4f}")
    print(f"Micro F1-Score:   {micro_f1_en:.4f}")
    print("==========================================\n")

    print("====== EFFICIENTNET PER-CLASS REPORT =====")
    print(classification_report(en_targets, en_preds, target_names=class_names, zero_division=0))
else:
    print("Please set the variable `MODEL_WEIGHTS_EFFICIENTNET` to your .pth file to evaluate.")

Running EfficientNet Inference: 100%|██████████| 157/157 [00:23<00:00,  6.69it/s]

========== EFFICIENTNET METRICS ==========
Macro Precision:  0.7718
Micro Precision:  0.7895
-------------------------------------
Macro Recall:     0.7319
Micro Recall:     0.7493
-------------------------------------
Macro F1-Score:   0.7502
Micro F1-Score:   0.7689

====== EFFICIENTNET PER-CLASS REPORT =====
                  precision    recall  f1-score   support

short sleeve top       0.80      0.80      0.80      2300
 long sleeve top       0.75      0.64      0.69      1388
          shorts       0.67      0.65      0.66       522
        trousers       0.82      0.85      0.83      1319
           skirt       0.83      0.73      0.77      1574

       micro avg       0.79      0.75      0.77      7103
       macro avg       0.77      0.73      0.75      7103
    weighted avg       0.79      0.75      0.77      7103
     samples avg       0.76      0.76      0.74      7103

